In [1]:
from pathlib import Path

import torch 
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
X_tensor = torch.tensor(X, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.long)

print("X shape:", X_tensor.shape)
print("y shape:", y_tensor.shape)

dataset = TensorDataset(X_tensor, y_tensor)

train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
)

In [ ]:
class SmallLanguageModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        context_length,
        embedding_dim=128,
        num_heads=4,
        num_layers=4,
        dropout=0.1,
    ):
        super().__init__()

        self.context_length = context_length

        # Converts token IDs into learned vectors.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
        )

        # Learns the position of each token.
        self.position_embedding = nn.Embedding(
            num_embeddings=context_length,
            embedding_dim=embedding_dim,
        )

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=embedding_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            transformer_layer,
            num_layers=num_layers,
        )

        self.final_norm = nn.LayerNorm(embedding_dim)

        # Converts hidden vectors into vocabulary predictions.
        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size,
            bias=False,
        )

        # Share weights between input embeddings and output layer.
        self.lm_head.weight = self.token_embedding.weight

    def forward(self, input_ids):
        batch_size, sequence_length = input_ids.shape

        if sequence_length > self.context_length:
            raise ValueError(
                f"Input length {sequence_length} exceeds "
                f"context length {self.context_length}"
            )

        positions = torch.arange(
            sequence_length,
            device=input_ids.device,
        )

        token_embeddings = self.token_embedding(input_ids)
        position_embeddings = self.position_embedding(positions)

        hidden_states = token_embeddings + position_embeddings

        # Prevent tokens from looking at future tokens.
        causal_mask = torch.triu(
            torch.ones(
                sequence_length,
                sequence_length,
                dtype=torch.bool,
                device=input_ids.device,
            ),
            diagonal=1,
        )

        hidden_states = self.transformer(
            hidden_states,
            mask=causal_mask,
        )

        hidden_states = self.final_norm(hidden_states)

        logits = self.lm_head(hidden_states)

        return logits

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = SmallLanguageModel(
    vocab_size=vocab_size,      # 1000 in your tokenizer
    context_length=100,
    embedding_dim=128,
    num_heads=4,
    num_layers=4,
).to(device)

print(model)
print("Device:", device)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.01,
)

loss_function = nn.CrossEntropyLoss()

num_epochs = 10

for epoch in range(num_epochs):
    model.train()

    total_loss = 0.0

    for input_ids, targets in train_loader:
        input_ids = input_ids.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        # Shape: batch, sequence, vocabulary
        logits = model(input_ids)

        loss = loss_function(
            logits.reshape(-1, vocab_size),
            targets.reshape(-1),
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Loss: {average_loss:.4f}"
    )